<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 4 (AI): RAG End to End — Grounding, Citations & a Real App

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Rebuild yesterday's pipeline with **LangChain** — and learn the industry name for every piece you wrote by hand
2. Join components with **LCEL**, the `|` pipe syntax
3. Load a **real PDF** and split it into chunks that carry `source` and `page`
4. Watch an ungrounded model **invent a policy**, confidently — then stop it
5. Build a chain that answers **only** from your documents, and says **"I don't know"** when the answer isn't there
6. Return **citations** — a real page number, taken from metadata
7. Fix retrieval when it fails: **k**, **MMR**, **query rewriting**, **hybrid search**, **reranking**
8. **Measure** retrieval with a golden set and hit-rate@k
9. Ship it as a **Gradio chat app** with a link you can open on your phone

> Yesterday you built retrieval. Today you turn it into something a person can actually use.

---

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q langchain langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf rank-bm25 fpdf2 gradio

In [ ]:
import os
import json
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# API key (typed securely - not shown on screen)
os.environ['OPENAI_API_KEY'] = getpass("Enter your OpenAI API Key: ")

MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

print("Setup complete")

> ⚠️ **Today needs an API key.** Yesterday's embeddings could run locally, but a *generation* step needs
> a chat model. If your key has no quota, pair up with someone for this session — every idea here is
> about the pipeline, not the provider.

---

## 2. Yesterday by Hand, Today by Name

Yesterday you wrote every part of a retrieval pipeline yourself. Each of those parts has a standard
name and a standard interface. That is essentially what **LangChain** is:

| What you wrote yesterday | What the industry calls it | Today |
|---|---|---|
| reading a file into a string | **Document Loader** | `PdfReader` → page text |
| `naive_chunks()` / the splitter | **Text Splitter** | `RecursiveCharacterTextSplitter` |
| `model.encode(...)` | **Embeddings** | `OpenAIEmbeddings` |
| the Chroma collection | **Vector Store** | `Chroma` |
| `search(query, n_results=3)` | **Retriever** | `vectorstore.as_retriever()` |
| the system prompt + chat call | **Prompt + Chat Model** | `ChatPromptTemplate` + `ChatOpenAI` |
| gluing them together by hand | **LCEL** — the `\|` pipe | `prompt \| llm \| parser` |

Nothing new is happening. The same six boxes, with names everyone recognises.

⚠️ **LangChain changed a lot in version 1.0.** If you find a tutorial using `RetrievalQA` or
`create_retrieval_chain`, it is written for the old version — those now live in a separate
`langchain-classic` package that is being retired. **The current way is LCEL**, which is what we use below.

In [ ]:
# LCEL: the "|" pipe joins components left to right, like a shell pipeline.
# Read it as: fill in the prompt -> send it to the model -> pull the plain text out.
llm = ChatOpenAI(model=MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template("Explain {thing} in one short sentence.")
chain = prompt | llm | StrOutputParser()

print(chain.invoke({"thing": "a vector database"}))

Three components, one chain, one `.invoke()`.

Every box in LangChain speaks the same two methods — `.invoke()` and `.stream()` — which is *why* they
can be piped together at all. Swap `llm` for a different provider and the rest of the chain doesn't change.

💡 Without `StrOutputParser()` you'd get a full message object back and have to reach into
`.content` yourself. The parser is just the "give me the plain string" step.

---

## 3. A Real Document

Yesterday's hook was: *"your college hands you a 200-page examination policy — build a bot that answers
questions about it."* Let's actually do that.

First we create the PDF, so everyone in the room is working from the identical document.
In your own project, this is simply a PDF you already have.

In [ ]:
# Build a small examination policy PDF - one page per section.
# (Text is plain ASCII on purpose: the built-in PDF fonts can't encode symbols like the rupee sign.)
from fpdf import FPDF

POLICY_PAGES = [
    ("Attendance", '''
A student must maintain at least 75 percent attendance in every course registered in a term.
Attendance is computed separately for lectures, tutorials and practicals.
A student whose attendance in a course falls below 75 percent is debarred from the end term
examination of that course and is awarded a grade of F.
Attendance records are published every week on the student portal. A correction request must be
raised within 7 days of publication, after which the record is treated as final.
Relaxation on medical grounds of up to 10 percent may be granted by the Dean of the school, on
production of a medical certificate submitted within 7 days of returning to class.
'''),
    ("Examinations and Grading", '''
Assessment in each course carries 100 marks: Continuous Assessment 40 marks, Mid Term Examination
20 marks, and End Term Examination 40 marks.
The passing mark is 40 percent in aggregate in each course.
The grade scale is: O for 90 and above, A plus for 80 to 89, A for 70 to 79, B for 60 to 69,
C for 50 to 59, D for 40 to 49, and F below 40.
The end term examination is 3 hours long. Calculators are not permitted unless the question paper
states otherwise.
Results are declared within 21 days of the last examination of the term.
'''),
    ("Re-evaluation and Re-appear", '''
A student may apply for re-evaluation of an end term answer script within 15 days from the date of
declaration of the result. Applications after this window are not accepted.
The re-evaluation fee is Rs. 1,000 per course and is payable online at the time of application.
If re-evaluation changes the total marks by 5 marks or more, the fee is refunded in full.
Re-appear examinations for backlog courses are held in the summer term, in the months of June and
July. A student gets a maximum of 3 attempts to clear a backlog course.
The registration fee for a re-appear examination is Rs. 2,500 per course.
'''),
    ("Academic Integrity", '''
Any violation of examination rules is recorded as an Unfair Means Case, referred to as a UMC.
Possession of a mobile phone or any smart device inside the examination hall is a UMC, whether or
not the device was used.
For a first offence, the answer script of that course is cancelled and the student is awarded F in
that course. For a second offence, the registration of the student is cancelled for the entire term.
A project report or dissertation with a similarity index above 20 percent must be resubmitted, and
the resubmitted work carries a penalty of 10 percent of the marks.
An appeal against a UMC decision may be filed with the UMC Committee within 10 days of the decision.
'''),
    ("Conduct of Examinations", '''
Students must report to the examination hall 30 minutes before the scheduled start time.
Entry is not permitted more than 15 minutes after the examination has begun.
The admit card and the institute identity card are both mandatory for entry. A student without an
admit card must obtain a duplicate from the Examination Cell before the examination starts.
No student may leave the examination hall during the first 45 minutes.
A make up examination is available to a student who misses an end term examination on documented
medical grounds, provided an application is made within 7 days of the missed examination.
'''),
    ("Registration and Course Load", '''
A student must register for a minimum of 15 credits and a maximum of 27 credits in a term.
Course registration opens 14 days before the start of the term and closes on the first day of
teaching. A student who does not register within this window is marked as not registered for the
term and may not appear in any examination of that term.
Courses may be added or dropped within 10 days of the start of teaching. A dropped course does not
appear on the transcript. A course withdrawn after this window is recorded as W on the transcript.
A student on academic probation may register for a maximum of 18 credits.
'''),
    ("Practical and Project Examinations", '''
A practical examination is of 3 hours duration and is assessed jointly by an internal examiner and
an external examiner appointed by the university.
The practical examination carries 60 marks, and the laboratory record and viva voce carry 40 marks.
A student must attend at least 75 percent of laboratory sessions to be eligible for the practical
examination.
The project report must be submitted 21 days before the end of the term. A late submission is
accepted up to 7 days beyond the deadline with a penalty of 10 percent of the project marks.
The viva voce for a project is conducted by a panel of at least two faculty members.
'''),
    ("Special Provisions", '''
A student with a certified benchmark disability is granted compensatory time of 20 minutes per hour
of examination.
A scribe may be permitted on written application to the Examination Cell at least 7 days before the
examination. The scribe must be of an academic level lower than that of the candidate.
A student representing the university in an inter university sports or cultural event is granted
attendance relaxation of up to 15 percent for the days of participation, on production of the
official team letter.
A student called for military or national service duty may defer an examination to the next
available session without penalty.
'''),
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

for title, body in POLICY_PAGES:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 15)
    pdf.cell(0, 12, title, new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 7, body.strip())

pdf.output("lpu_exam_policy.pdf")
print("Wrote lpu_exam_policy.pdf")

In [ ]:
# Read the PDF back - one string per page.
from pypdf import PdfReader

reader = PdfReader("lpu_exam_policy.pdf")
page_texts = [page.extract_text() for page in reader.pages]

print(f"Pages: {len(page_texts)}")
print()
print(page_texts[0][:320])

That is the whole of "document loading": a PDF in, a list of page strings out.

The important part is what came with it — **which page each string came from**. Hold on to that number.
It is the difference between an answer you have to trust and an answer you can *check*.

🧑‍🏫 Real PDFs are messier than this one: scanned pages have no text layer at all (you need OCR), and
two-column layouts often extract in the wrong reading order. Always print the extracted text before you
trust it.

---

## 4. Split into Chunks — carrying metadata

Same splitter you met yesterday, with one addition: each chunk gets **metadata** attached.

In [ ]:
# "Recursive character" = try paragraph breaks first, then sentences, then words -
# whatever keeps the chunk under the size limit without cutting mid-word.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
)

# One metadata dict per page, so every chunk inherits the page it came from.
chunks = splitter.create_documents(
    page_texts,
    metadatas=[{"source": "lpu_exam_policy.pdf", "page": i + 1} for i in range(len(page_texts))],
)

# Give each chunk a number too - we'll use it for hybrid search later.
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"{len(page_texts)} pages -> {len(chunks)} chunks")

In [ ]:
# A LangChain Document is just text + metadata. Change the index and re-run.
chunk = chunks[6]

print(chunk.page_content)
print()
print("metadata:", chunk.metadata)

`page_content` and `metadata` — that is the entire `Document` class.

Everything downstream now travels as a `Document`, which is why the page number survives all the way
from the PDF to the citation at the bottom of the answer.

💡 **Attach metadata at split time, not later.** Once chunks are embedded and stored, working out which
page a chunk came from means re-doing the work.

---

## 5. Embed, Store, Retrieve

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# One line does yesterday's three steps: embed every chunk, store the vectors,
# and keep the text and metadata alongside them.
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="lpu_exam_policy",
)

# A retriever is a vector store with one job: question in, Documents out.
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Indexed", len(chunks), "chunks")

⚠️ **Collection names are validated.** Chroma requires 3–512 characters from `a-z A-Z 0-9 . _ -`,
starting and ending with a letter or number. `"t1"` or `"my docs"` will raise an error.

In [ ]:
# Question in, Documents out. Change the question and re-run.
question = "How much attendance do I need?"

docs = retriever.invoke(question)

print(f"{len(docs)} chunks retrieved. The top one is from page {docs[0].metadata['page']}:")
print()
print(docs[0].page_content[:400])

**Try these one at a time**, changing `question` above:

- `"What is the passing mark?"`
- `"What happens if I bring my phone into the exam hall?"`
- `"When are backlog exams held?"`

🧑‍🏫 The `retriever` is an *interface*, not a database. Swap Chroma for pgvector, Pinecone or FAISS and
every line after this one stays exactly the same. That is the real payoff of the abstraction.

---

## 6. First, Watch It Lie

Before building the good version, see the problem clearly. Here is the model with **no context and no
rules** — just answering from what it absorbed during training.

In [ ]:
# No retrieval, no grounding - the model answering from memory.
question = "What is the re-evaluation fee at LPU, and how many days do I have to apply?"

print(llm.invoke(question).content)

Read that answer carefully. It is fluent, it is specific, it is formatted like a policy — and it is
**invented**. The model has never seen this document.

This is the failure that makes RAG necessary, and it is worth naming precisely, because RAG systems
break in two different ways and each has a different fix:

| Failure | What happened | How you spot it | The fix |
|---|---|---|---|
| **Retrieval failure** | the right chunk was never found | print the retrieved chunks — the answer isn't in them | section 9: k, rewriting, reranking |
| **Generation failure** | the right chunk was found, the answer still went wrong | the answer isn't in them | section 7: the prompt contract |

> 🔍 **The diagnostic:** before blaming the model, *look at what you handed it.* Almost every "the LLM is
> hallucinating" bug turns out to be a retrieval bug.

---

## 7. Grounding — the chain that says "I don't know"

Now the real thing. Two rules do most of the work: **use only the context**, and **refuse when it isn't there**.

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You answer questions about the LPU examination policy.\n"
     "Use ONLY the context below. Do not use any other knowledge.\n"
     "If the context does not contain the answer, reply exactly: "
     "I don't know based on the policy document.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])


def format_docs(docs):
    """Turn a list of retrieved Documents into one plain string for the prompt."""
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# The whole RAG pipeline, as one chain.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the re-evaluation fee, and how many days do I have to apply?"))

Compare that with section 6. Same question, same model — the only thing that changed is that it was
handed the page and told to stay on it.

Read the chain top to bottom:

```
                 "What is the re-evaluation fee?"
                                |
        +-----------------------+-----------------------+
        |                                               |
   retriever                                     RunnablePassthrough
   (find 3 chunks)                               (keep the question as-is)
        |                                               |
   format_docs                                          |
        |                                               |
        +------------------> RAG_PROMPT <---------------+
                                 |
                                llm            temperature=0
                                 |
                          StrOutputParser
                                 |
                              answer
```

The dict at the front is the trick worth remembering: **each key is filled in parallel**, and the
result is exactly the `{context}` and `{question}` the prompt is asking for.

In [ ]:
# Yesterday, this question still returned three confident chunks. Now watch.
print(rag_chain.invoke("What is the hostel fee at LPU?"))

💡 **That refusal is the feature.** Retrieval on its own has no notion of "nothing here is relevant" — it
always returns its top-k. The *prompt* is what turns "here are the 3 closest chunks" into "this document
doesn't answer your question."

A system that says "I don't know" 5% of the time is worth far more than one that is confidently wrong
5% of the time — because nobody can tell which 5% they're reading.

🧑‍🏫 Note `temperature=0` on the model. For RAG you want reading, not creative writing.

---

## 8. Citations

An answer a student can't verify is an answer they have to take on faith. Citations fix that — and they
come from the **metadata**, never from the model.

In [ ]:
# .assign() runs the answer chain but keeps the retrieved Documents alongside it,
# instead of throwing them away like rag_chain does.
rag_with_sources = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough(),
).assign(
    answer=(lambda x: {"context": format_docs(x["context"]), "question": x["question"]})
           | RAG_PROMPT
           | llm
           | StrOutputParser()
)

result = rag_with_sources.invoke("How many days do I have to apply for re-evaluation?")

print(result["answer"])
print()
print("Sources:")
for doc in result["context"]:
    print(f"  - {doc.metadata['source']}, page {doc.metadata['page']}")

The result is a dictionary with three keys: `question`, `context` (the Documents) and `answer`.

⚠️ **Never ask the model to produce the citation itself.** A model asked to "cite your source" will
happily invent a plausible page number — that is exactly the behaviour you're trying to eliminate.
The page number above was carried from the PDF, through the splitter, through the vector store, and
printed by *your* code. The model never touched it.

💡 This is also your debugging view. If an answer looks wrong, print `result["context"]` and you will
immediately see whether retrieval or generation is at fault.

---

## 9. When Retrieval Fails — the fix kit

Grounding stops the model inventing things. It does **not** help when the right chunk was never retrieved —
then a well-behaved system politely says "I don't know" about something that is on page 3.

Start every investigation the same way: **look at what was retrieved.**

In [ ]:
# The habit that saves hours: inspect the chunks before blaming the model.
question = "What happens if I am caught using unfair means?"

docs = retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    print(f"[{i}] page {doc.metadata['page']}: {doc.page_content[:110]}...")
    print()

### 9.1 Turn the knobs — `k` and MMR

The cheapest fix first: retrieve **more**.

In [ ]:
# k = how many chunks come back. Change it and re-run.
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

docs = wide_retriever.invoke("What happens if I am caught using unfair means?")

print("pages retrieved:", [d.metadata["page"] for d in docs])

More chunks means better odds of catching the right one — and a longer, costlier, noisier prompt.
Research on ["Lost in the Middle"](https://arxiv.org/abs/2307.03172) shows models attend best to the
**start and end** of their context, so burying the good chunk in the middle of ten others can make things
*worse*. Retrieve broad, then narrow (that's reranking, in 9.4).

In [ ]:
# MMR picks chunks that are relevant AND different from one another, so you don't
# spend all three slots on near-duplicate paragraphs.
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10},
)

docs = mmr_retriever.invoke("What are the rules about attendance and examinations?")

print("pages retrieved:", [d.metadata["page"] for d in docs])

### 9.2 Query rewriting

Users don't type search queries. They type *questions* — vague, conversational, full of pronouns.
Rewrite before retrieving.

In [ ]:
REWRITE_PROMPT = ChatPromptTemplate.from_template(
    "Rewrite the question as a short, keyword-rich search query for a university policy document.\n"
    "Return ONLY the query.\n\n"
    "Question: {question}"
)
rewriter = REWRITE_PROMPT | llm | StrOutputParser()

vague = "i missed loads of classes, am i in trouble?"

print("Original: ", vague)
print("Rewritten:", rewriter.invoke({"question": vague}))

In [ ]:
# Does the rewrite actually retrieve better? Compare the pages each one finds.
print("raw       ->", [d.metadata["page"] for d in retriever.invoke(vague)])
print("rewritten ->", [d.metadata["page"] for d in retriever.invoke(rewriter.invoke({"question": vague}))])

### 9.3 Hybrid search  `[Extended]`

Vector search is strong on meaning and weak on **exact strings**. Ask it for "UMC" and it may hand you
paragraphs that are *about* misconduct without containing the term. Keyword search (BM25) has the
opposite strengths — so production systems run both and merge the rankings.

In [ ]:
from rank_bm25 import BM25Okapi

chunk_texts = [c.page_content for c in chunks]
bm25 = BM25Okapi([t.lower().split() for t in chunk_texts])


def hybrid_search(query, k=3):
    """Merge two rankings with Reciprocal Rank Fusion: score = sum of 1 / (60 + rank)."""
    scores = {}

    # ranking 1 - meaning
    for rank, doc in enumerate(vectorstore.similarity_search(query, k=10)):
        idx = doc.metadata["chunk_id"]
        scores[idx] = scores.get(idx, 0) + 1 / (60 + rank)

    # ranking 2 - exact words
    for rank, idx in enumerate(bm25.get_scores(query.lower().split()).argsort()[::-1][:10]):
        scores[idx] = scores.get(idx, 0) + 1 / (60 + rank)

    best = sorted(scores, key=scores.get, reverse=True)[:k]
    return [chunks[i] for i in best]

In [ ]:
# A query with an exact term in it - change it and re-run.
query = "UMC appeal committee"

print("vector only ->", [d.metadata["page"] for d in vectorstore.similarity_search(query, k=3)])
print("hybrid      ->", [d.metadata["page"] for d in hybrid_search(query, k=3)])

💡 **Reciprocal Rank Fusion** needs no tuning and no score normalisation — it only cares *what position*
each result reached in each list. A chunk that both methods rank highly wins; a chunk only one method
loves still gets a chance. The `60` is a standard constant that stops rank 1 dominating everything.

### 9.4 Reranking  `[Extended]`

The embedding model encoded every chunk **before it ever saw your question** — that is what makes search
fast, and also what makes it approximate. A **reranker** looks at the question and a chunk *together* and
scores that pair properly.

```
retrieve 6 (fast, approximate)  ->  rerank (slow, accurate)  ->  keep top 3  ->  LLM
```

In [ ]:
RERANK_PROMPT = ChatPromptTemplate.from_template(
    "Question: {question}\n\n"
    "Documents:\n{docs}\n\n"
    "Reorder the documents from most to least relevant to the question.\n"
    "Return ONLY a JSON array of document numbers, for example [3, 1, 2]."
)
reranker = RERANK_PROMPT | llm | StrOutputParser()

question = "What happens if I am caught using unfair means?"

# Retrieve broad...
candidates = vectorstore.similarity_search(question, k=6)
listing = "\n".join(f"[{i + 1}] {d.page_content[:180]}" for i, d in enumerate(candidates))

# ...then let the model put them in order.
order = json.loads(reranker.invoke({"question": question, "docs": listing}))

print("original pages:", [d.metadata["page"] for d in candidates])
print("reranked order:", order)
print("new top chunk is from page", candidates[order[0] - 1].metadata["page"])

🧑‍🏫 **In production you would not use an LLM for this.** A dedicated **cross-encoder** is 10–100× faster
and cheaper: [`cross-encoder/ms-marco-MiniLM-L-6-v2`](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)
is free and runs locally, and [Cohere Rerank](https://docs.cohere.com/docs/rerank) is the common hosted
option (about $0.0025 per search of up to 100 documents). We use the LLM here because you already have
the key — the *idea* is identical.

### The fix kit, summarised

| Symptom | Reach for | Cost |
|---|---|---|
| right chunk just missed the cut | bigger `k` | longer prompt |
| three chunks all say the same thing | **MMR** | none |
| question is vague or conversational | **query rewriting** | +1 LLM call |
| exact terms, codes, names missed | **hybrid search** | none |
| right chunk retrieved but ranked 5th | **reranking** | +1 call / +latency |
| nothing relevant exists | the grounding prompt (section 7) | none |

---

## 10. Measure It

Every knob above is a guess until you measure it. The cheapest useful measurement in RAG: take a handful
of real questions, write down a fact that **must** appear in the retrieved chunks, and count how often it does.

That's a **golden set**, and 6 questions written in ten minutes will teach you more than any amount of
staring at the code.

In [ ]:
# A golden set: a real question + a fact the retrieved chunks MUST contain.
GOLDEN = [
    ("How much attendance do I need?",                    "75 percent"),
    ("What is the passing mark in a course?",             "40 percent"),
    ("How long do I have to apply for re-evaluation?",    "15 days"),
    ("What is the re-evaluation fee?",                    "1,000"),
    ("What if I am caught with a phone in the exam hall?", "mobile phone"),
    ("When are backlog exams held?",                      "summer"),
]


def hit_rate(search_fn, k=3):
    """Fraction of questions whose expected fact appears somewhere in the top-k chunks."""
    hits = 0
    for question, expected in GOLDEN:
        retrieved = " ".join(d.page_content.lower() for d in search_fn(question, k))
        hits += expected.lower() in retrieved
    return hits / len(GOLDEN)

In [ ]:
# One number for the whole retrieval step. Change k and re-run.
k = 3

score = hit_rate(lambda q, n: vectorstore.similarity_search(q, k=n), k=k)

print(f"hit rate @{k}: {score:.0%}")

Now you can answer questions that were previously matters of opinion:

- Does `k=1` still work? (drop `k` and re-run)
- Is hybrid search better *here*? (pass `hybrid_search` instead of `similarity_search`)
- Was `chunk_size=500` a good choice? (change it in section 4, re-run everything below it)

🧑‍🏫 **This is the honest answer to "what chunk size should I use?"** — you measure it. Not vibes,
not a blog post. And notice this measures **retrieval only**: it says nothing about whether the final
*answer* was good. Judging answer quality (LLM-as-judge, faithfulness, RAGAS) is Day 5.

⚠️ A golden set of 6 is a teaching size. Aim for 30–100 real user questions before you trust the number.

---

## 11. Ship It — a chat app

A notebook cell is not a product. The last step is a chat interface — and chat introduces one genuinely
new problem: **follow-up questions don't retrieve well on their own.**

> "How much attendance do I need?" → *75 percent*
> "**And what if I was ill?**" ← embed *that* and you will retrieve nothing useful.

The fix is the rewriter from section 9.2, now given the conversation so far.

In [ ]:
def rag_answer(message, history):
    """One chat turn: resolve the follow-up, retrieve, ground, then cite."""

    # 1) Turn a follow-up into a standalone search query using what was said before.
    if history:
        recent = "\n".join(f"{m['role']}: {m['content']}" for m in history[-4:])
        search_query = rewriter.invoke({"question": f"{recent}\nFollow-up: {message}"})
    else:
        search_query = message

    # 2) Retrieve on the rewritten query, but answer the question the user actually asked.
    docs = retriever.invoke(search_query)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(docs), "question": message}
    )

    # 3) Cite from metadata.
    pages = sorted({d.metadata["page"] for d in docs})
    return f"{answer}\n\n*Source: lpu_exam_policy.pdf - page {', '.join(str(p) for p in pages)}*"

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    rag_answer,
    type="messages",
    title="LPU Examination Policy Assistant",
    description="Ask about attendance, grading, re-evaluation, or exam conduct. Answers come only from the policy PDF.",
    examples=[
        "How much attendance do I need?",
        "What is the passing mark in a course?",
        "How do I apply for re-evaluation?",
        "What is the hostel fee?",
    ],
)

demo.launch(share=True)

Open the `gradio.live` link on your phone. That is a working document assistant, and every part of it is
something you built today.

**Try the last example — "What is the hostel fee?"** It is in the list deliberately. The policy says
nothing about hostel fees, and a good assistant should say so rather than guess.

Then try a follow-up: ask *"How much attendance do I need?"*, and then just *"and what if I was ill?"*
Watch the rewriter turn that into a real search query.

---

## 12. Exercises

Fill in the blanks (`___`) and run each cell.

### Q1: Your first LCEL chain

Build a three-step chain that turns a topic into a one-line exam tip.

In [ ]:
# Hint: the pipe order is always prompt | llm | parser.
#       ChatPromptTemplate.from_template uses {braces} for variables.

tip_prompt = ChatPromptTemplate.from_template("Give one short exam tip about {topic}.")

tip_chain = tip_prompt | ___ | StrOutputParser()

print(tip_chain.invoke({"___": "time management"}))

### Q2: Retrieve and read the metadata

Retrieve the chunks for a question of your own and print which page each came from.

In [ ]:
# Hint: retriever.invoke(question) returns a list of Documents.
#       Each Document has .page_content and .metadata

my_question = "Can I leave the exam hall early?"

my_docs = retriever.___(my_question)

print("pages:", [d.metadata["___"] for d in my_docs])
print()
print(my_docs[0].page_content[:200])

### Q3: Make it refuse

Write a grounded chain of your own, then ask it something the policy does not cover.

In [ ]:
# Hint: the two rules are "use ONLY the context" and "say you don't know otherwise".
#       The dict keys must match the {variables} in your prompt.

MY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context. If it is not there, say you don't know.\n\n{context}"),
    ("human", "{question}"),
])

my_chain = (
    {"context": retriever | ___, "question": RunnablePassthrough()}
    | MY_PROMPT
    | llm
    | ___
)

print(my_chain.invoke("What is the passing mark?"))          # answerable
print(my_chain.invoke("___"))                                 # ask something NOT in the policy

### Q4: Measure a change

Add one question of your own to the golden set, then compare two values of `k`.

In [ ]:
# Hint: hit_rate(search_fn, k) - the search_fn takes (query, k) and returns Documents.

GOLDEN.append(("How long is the end term examination?", "___"))   # a fact that must be retrieved

low  = hit_rate(lambda q, n: vectorstore.similarity_search(q, k=n), k=___)
high = hit_rate(lambda q, n: vectorstore.similarity_search(q, k=n), k=___)

print(f"low k:  {low:.0%}")
print(f"high k: {high:.0%}")

### Q5: Cite your sources

Return an answer together with the pages it came from.

In [ ]:
# Hint: rag_with_sources.invoke(question) returns a dict with "answer" and "context".

output = rag_with_sources.___("What happens on a second unfair means offence?")

print(output["___"])
print()
print("Cited pages:", sorted({d.metadata["page"] for d in output["___"]}))

---

### ✅ Recap

| Idea | The one-liner |
|---|---|
| **LCEL** | `prompt \| llm \| parser` — every component speaks `.invoke()`, so they pipe together |
| **Document** | `page_content` + `metadata`; the metadata is what makes citation possible |
| **Retriever** | question in, Documents out — swap the vector store, the chain doesn't change |
| **Grounding** | "use ONLY the context" + "say you don't know" — two rules, most of the value |
| **`temperature=0`** | for RAG you want the model reading, not inventing |
| **Citations** | come from *your* metadata; a model asked to cite will invent page numbers |
| **The diagnostic** | print the retrieved chunks before blaming the model |
| **Retrieval failure** | right chunk never found → `k`, MMR, rewriting, hybrid, reranking |
| **Generation failure** | right chunk found, answer still wrong → fix the prompt |
| **Hybrid search** | vectors find meaning, BM25 finds exact strings; merge with RRF |
| **Reranking** | retrieve broad and approximate, then score query+chunk together and keep the best |
| **Golden set** | a few real questions + expected facts = the only honest way to tune |

### 🏠 Homework

1. **Swap the document.** Replace `lpu_exam_policy.pdf` with a PDF you actually care about — a syllabus,
   a manual, a research paper. Everything below section 3 should work unchanged.
2. **Write a golden set of 10** questions for your document, and record the hit rate at `k=1`, `3` and `5`.
3. **Break it on purpose.** Find one question where retrieval fails, then fix it with exactly one
   technique from section 9 — and write down which one and why.

### 📚 Resources

- [LangChain — LCEL and Runnables](https://python.langchain.com/docs/concepts/lcel/)
- [LangChain — retrievers](https://python.langchain.com/docs/concepts/retrievers/)
- [Lost in the Middle (Liu et al., 2023)](https://arxiv.org/abs/2307.03172) — why position in the context matters
- [Query Rewriting for Retrieval-Augmented LLMs (Ma et al., 2023)](https://arxiv.org/abs/2305.14283)
- [Cohere Rerank](https://docs.cohere.com/docs/rerank) · [free local cross-encoder](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2)
- [RAGAS](https://docs.ragas.io/) — evaluation at production scale *(Day 5)*

---

**Next — Day 5:** agents that decide *which* tool to use and when, plus production GenAI: cost, evaluation and responsible AI.